# Protocol 67 LoRA Training Notebook

This Colab notebook trains the Protocol 67 slang-to-English LoRA adapter on a GPU runtime.

`train_lorav2.py` uses GPU-oriented dependencies such as PyTorch CUDA and bitsandbytes. Colab with a GPU, such as a Tesla T4, is the recommended environment for this training run.

Expected output:

```text
model_service/outputs/llama3b-slang-lora
```

After training finishes, copy that output folder to Google Drive so it can be downloaded or reused later.


## Adapter Folder Structure

The notebook uses this adapter folder name and location:

```text
model_service/outputs/llama3b-slang-lora
```

Keep the adapter files directly inside that folder, for example `adapter_config.json`, `adapter_model.safetensors`, and `tokenizer.json`. Avoid adding an extra nested `llama3b-slang-lora` folder inside it.

The Google Drive copy command saves the folder under:

```text
`/content/drive/MyDrive/protocol67-models/llama3b-slang-lora` by default, or the configured `DRIVE_ADAPTER_DIR` if you change it
```

You can use your own Drive path or adapter folder name, but update the copy commands and any later inference notebook paths to match your chosen structure.


In [ ]:
# Change this if you want to save the trained adapter somewhere else in Google Drive.
DRIVE_MODEL_DIR = "/content/drive/MyDrive/protocol67-models"
ADAPTER_NAME = "llama3b-slang-lora"
DRIVE_ADAPTER_DIR = f"{DRIVE_MODEL_DIR}/{ADAPTER_NAME}"
DRIVE_ADAPTER_DIR


## 1. Set Colab Runtime Type

Before running the training cells, switch Colab to a GPU runtime:

```text
Runtime -> Change runtime type -> Hardware accelerator -> GPU
```

After the runtime reconnects, run `nvidia-smi` to confirm that a GPU is available.


In [ ]:
!nvidia-smi


## 2. Clone The Repository

Run this first in a fresh Colab runtime. It downloads the project and moves the notebook session into the repository root. The later commands assume the current folder contains `backend`, `data_pipeline`, `frontend`, and `model_service`.


In [2]:
!git clone https://github.com/hsyen78444/Protocol-67.git
%cd Protocol-67

Cloning into 'Protocol-67'...
remote: Enumerating objects: 355, done.
remote: Counting objects: 100% (355/355), done.
remote: Compressing objects: 100% (227/227), done.
remote: Total 355 (delta 149), reused 311 (delta 119), pack-reused 0 (from 0)
Receiving objects: 100% (355/355), 1.48 MiB | 16.33 MiB/s, done.
Resolving deltas: 100% (149/149), done.
/content/Protocol-67


## 3. Install Training Dependencies

Install the model-service requirements and `bitsandbytes`. The LoRA training script uses Transformers, PEFT, TRL, Datasets, Accelerate, PyTorch, and 4-bit quantization support.


In [4]:
!python -m pip install --upgrade pip
!python -m pip install -r model_service/requirements.txt
!python -m pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 21.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 25.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 68.1 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [trl]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.9 MB/s  0:00:01


## 4. Authenticate With Hugging Face

The base model is `meta-llama/Llama-3.2-3B-Instruct`, which is gated. Before running the login cell:

1. Create or sign in to a Hugging Face account: https://huggingface.co/join
2. Generate a read token: https://huggingface.co/settings/tokens
3. Request access for the Llama model: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

Use a token with `Read` permission. When the notebook prompts for the token, paste it into the hidden input. If asked whether to add the token as a git credential, choose `N` unless you specifically need git credential storage.

If training fails with a gated repository or 403 error, confirm that model access has been granted for the same Hugging Face account used to create the token.


In [6]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: N
Token is valid (permission: read).
The token `protocol67` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `protocol67`


## 5. Verify The Processed Dataset

This checks that the processed train, validation, and test JSONL files are present and readable. The training script uses `data_pipeline/data/processed/train.jsonl` and `data_pipeline/data/processed/validation.jsonl`.


In [7]:
!python -m model_service.scripts.prepare_dataset

train: 4115 rows
                                                                       instruction                                                                                                                                                    input                                                                                                                                   output                                                                                                                                                               metadata
Translate the following internet slang or brainrot text into clear formal English. 3am brain just hits different staring at the ceiling questioning everything? the existential dread goes hard in the silence send happy thoughts or memes The existential dread that occasionally creeps in during late-night hours often feels amplified by the surrounding silence and darkness.               {'source': 'new_raw_parallel', 'platform': 'synthetic', 

## 6. Mount Google Drive

Mount Drive so the final trained adapter can be copied outside the temporary Colab runtime after training. Mounting Drive by itself does not save in-progress checkpoints; the notebook copies the output folder to Drive after training completes.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!mkdir -p "{DRIVE_MODEL_DIR}"


## 7. Optional Access Test

This cell tests whether the current Hugging Face token can access the Llama tokenizer. If this fails with a gated repository error, request/accept access for the model before training.


In [13]:
!python -c "from transformers import AutoTokenizer; AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B-Instruct'); print('access works')"

config.json: 100% 878/878 [00:00<00:00, 4.53MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 40.7MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 24.2MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.71MB/s]
access works


## 8. Train The LoRA Adapter

Run the latest training script. It loads Llama 3.2 3B with 4-bit quantization, formats the processed dataset into chat-style prompts, trains a LoRA adapter, evaluates every 100 steps, and saves checkpoints under `model_service/outputs/llama3b-slang-lora`.

Training may take a few hours on a free Colab GPU. The script saves checkpoints every 100 steps.


In [14]:
!python -m model_service.scripts.prepare_dataset
!python -m model_service.scripts.train_lorav2

train: 4115 rows
                                                                       instruction                                                                                                                                                    input                                                                                                                                   output                                                                                                                                                               metadata
Translate the following internet slang or brainrot text into clear formal English. 3am brain just hits different staring at the ceiling questioning everything? the existential dread goes hard in the silence send happy thoughts or memes The existential dread that occasionally creeps in during late-night hours often feels amplified by the surrounding silence and darkness.               {'source': 'new_raw_parallel', 'platform': 'synthetic', 

## 9. Save The Trained Adapter To Drive

After training finishes, copy the adapter folder to Google Drive. The saved folder can later be downloaded and placed back under `model_service/outputs/llama3b-slang-lora` in a local copy of the project.


In [15]:
!cp -r model_service/outputs/llama3b-slang-lora "{DRIVE_MODEL_DIR}"/
